In [6]:
import pandas as pd
import duckdb
import rank_bm25
import nltk

In [25]:
con = duckdb.connect()

df = con.execute("""
    SELECT *
    FROM read_parquet('../data/processed/merged.parquet')
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details
0,4.0,Keeps leather from drying out,I have a leather couch and because we live in ...,True,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,None,[],Howard Products,"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ..."
1,5.0,Five Stars,Grandson has had very good results with this p...,True,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,None,[],Yes To,"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro..."
2,5.0,Good Eye Patch,It is good. No problems. Eye patch is as adver...,True,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,None,[],Levine Health Products,"{""Manufacturer"": ""Levine Health Products""}"
3,5.0,Wonderful and super realistic,I have trichotillomania which is a hair pullin...,True,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,None,[],Cherioll,"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""..."
4,5.0,Missing Review.......... It’s lost at Amazon S...,Where did my review go? I used your take a pic...,True,Precision Plunger Bars for Cartridge Grips – 9...,4.3,None,[The Precision Plunger Bars are designed to wo...,Precision,"{""UPC"": ""644287689178""}"


In [5]:
df["text"] = df["text"].fillna("").astype(str)
df.head()

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details
0,4.0,Keeps leather from drying out,I have a leather couch and because we live in ...,True,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,None,[],Howard Products,"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ..."
1,5.0,Five Stars,Grandson has had very good results with this p...,True,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,None,[],Yes To,"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro..."
2,5.0,Good Eye Patch,It is good. No problems. Eye patch is as adver...,True,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,None,[],Levine Health Products,"{""Manufacturer"": ""Levine Health Products""}"
3,5.0,Wonderful and super realistic,I have trichotillomania which is a hair pullin...,True,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,None,[],Cherioll,"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""..."
4,5.0,Missing Review.......... It’s lost at Amazon S...,Where did my review go? I used your take a pic...,True,Precision Plunger Bars for Cartridge Grips – 9...,4.3,None,[The Precision Plunger Bars are designed to wo...,Precision,"{""UPC"": ""644287689178""}"


In [7]:
nltk.download("stopwords")

from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/arafat/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Writing the preprocessing function to tokenize the text

In [17]:
import re
def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove whitespace
    tokens = text.split()

    # remove stop words
    tokens = [word for word in tokens if word not in stop_words]

    return tokens

In [18]:
df["tokens"] = df["text"].apply(preprocess_text)

In [19]:
df.head()

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details,tokens
0,4.0,Keeps leather from drying out,I have a leather couch and because we live in ...,True,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,None,[],Howard Products,"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ...","[leather, couch, live, area, air, conditioning..."
1,5.0,Five Stars,Grandson has had very good results with this p...,True,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,None,[],Yes To,"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro...","[grandson, good, results, product, acne]"
2,5.0,Good Eye Patch,It is good. No problems. Eye patch is as adver...,True,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,None,[],Levine Health Products,"{""Manufacturer"": ""Levine Health Products""}","[good, problems, eye, patch, advertised, would..."
3,5.0,Wonderful and super realistic,I have trichotillomania which is a hair pullin...,True,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,None,[],Cherioll,"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""...","[trichotillomania, hair, pulling, disorder, ey..."
4,5.0,Missing Review.......... It’s lost at Amazon S...,Where did my review go? I used your take a pic...,True,Precision Plunger Bars for Cartridge Grips – 9...,4.3,None,[The Precision Plunger Bars are designed to wo...,Precision,"{""UPC"": ""644287689178""}","[review, go, used, take, picture, link, came, ..."


## Build BM25

In [11]:
from rank_bm25 import BM25Okapi

In [20]:
tokenized_corpus = df["tokens"].tolist()
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
import os
data_dir = "../data/processed"
os.makedirs(data_dir, exist_ok=True)
df.to_parquet(os.path.join(data_dir, "tokenized_docs.parquet"), index=False)


In [24]:
import pickle
os.makedirs("../models", exist_ok=True)
with open(os.path.join("../models", "bm25_model.pkl"), "wb") as f:
    pickle.dump(bm25, f)

with open(os.path.join("../data/processed", "tokenized_corpus.pkl"), "wb") as f:
    pickle.dump(tokenized_corpus, f)

## Search function

In [26]:
def search(query, top_k=3):
    tokenized_query = preprocess_text(query)
    scores = bm25.get_scores(tokenized_query)
    
    docs_copy = df.copy()
    docs_copy["score"] = scores
    
    return docs_copy.sort_values("score", ascending=False).head(top_k)[["title", "text", "score","rating"]]

In [27]:
search("good battery life")

,title,text,score,rating
684744,Great shaver,"great shaver, good battery life, battery life...",19.842759,5.0
127376,It's ok 👌🙂 not to bad,Not a problem with it battery life is good,18.737010,5.0
267740,Great Shaver all round cuts to the edge,Battery life good very easy to clean,17.993469,5.0
